# CLIP Linear Probe Baseline

Notebook này dùng data loader chung trong `data_loader/` để train/test baseline:

```text
image -> frozen CLIP image encoder -> image embedding -> Logistic Regression -> real/fake
```

CLIP được freeze hoàn toàn. Chỉ classifier tuyến tính ở tầng cuối được train.

In [ ]:
%pip install -q datasets open_clip_torch scikit-learn matplotlib tqdm

## 1. Import và cấu hình

In [ ]:
from pathlib import Path
import json
import sys
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data_loader").exists():
    parents = [PROJECT_ROOT.parent, PROJECT_ROOT.parent.parent]
    for candidate in parents:
        if (candidate / "data_loader").exists():
            PROJECT_ROOT = candidate
            break

sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT =", PROJECT_ROOT)

from data_loader import (
    TinyGenImageDataset,
    TinyGenImageIterableDataset,
    TinyGenImageSplitConfig,
    build_tiny_genimage_splits,
    collate_unified_batch,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

## 2. Chọn 4 experiment case

In [ ]:
# Notebook sẽ chạy lần lượt cả 4 trường hợp:
# 1. combined: train tất cả generator, test tất cả generator
# 2. in_domain: train/test real + fake của một generator
# 3. cross_generator: leave-one-generator-out
# 4. train_one_generator: train một generator, test tất cả generator

# True: đọc lazy từ Hugging Face, không tải full dataset về cache local trước.
# False: tải/cache dataset, train lặp lại nhanh hơn.
STREAMING = True
BALANCE_REAL = True
CACHE_DIR = None

# Dùng để test nhanh notebook trước. Đặt None để chạy full split.
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None

CLIP_MODEL_NAME = "ViT-B-32"
CLIP_PRETRAINED = "openai"
BATCH_SIZE = 64
NUM_WORKERS = 0
RANDOM_SEED = 42

EXPERIMENT_CONFIGS = [
    {"name": "combined", "eval_case": "combined"},
    {"name": "in_domain_biggan", "eval_case": "in_domain", "generator": "BigGAN"},
    {"name": "cross_generator_glide", "eval_case": "cross_generator", "heldout_generator": "GLIDE"},
    {"name": "train_one_generator_biggan", "eval_case": "train_one_generator", "base_generator": "BigGAN"},
]

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "clip_linear_probe"
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
print("RUN_ID =", RUN_ID)
print("Số experiment:", len(EXPERIMENT_CONFIGS))

## 3. Helper build split

In [ ]:
def build_splits_for_experiment(exp):
    config = TinyGenImageSplitConfig(
        eval_case=exp["eval_case"],
        generator=exp.get("generator"),
        heldout_generator=exp.get("heldout_generator"),
        base_generator=exp.get("base_generator"),
        cache_dir=CACHE_DIR,
        streaming=STREAMING,
        balance_real=BALANCE_REAL,
        seed=RANDOM_SEED,
    )
    splits = build_tiny_genimage_splits(config)
    print("\n===", exp["name"], "===")
    print(splits["notes"])
    print("streaming:", splits.get("streaming"))
    print("balance_real:", splits.get("balance_real"))
    print("train real/fake:", splits.get("train_real_count"), splits.get("train_fake_count"))
    print("eval real/fake:", splits.get("eval_real_count"), splits.get("eval_fake_count"))
    print("train rows:", splits.get("train_rows") if STREAMING else len(splits["train"]))
    print("eval rows:", splits.get("eval_rows") if STREAMING else len(splits["eval"]))
    return splits

## 4. Load CLIP frozen encoder

In [ ]:
import open_clip

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME,
    pretrained=CLIP_PRETRAINED,
    device=DEVICE,
)
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad = False

print("Loaded CLIP:", CLIP_MODEL_NAME, CLIP_PRETRAINED)

## 5. Helper tạo Dataset/DataLoader

In [ ]:
def maybe_limit_dataset(dataset, max_samples, streaming):
    if max_samples is None:
        return dataset
    return dataset.take(max_samples) if streaming else dataset.select(range(min(max_samples, len(dataset))))


def build_loaders(splits):
    DatasetClass = TinyGenImageIterableDataset if splits.get("streaming") else TinyGenImageDataset
    train_source = maybe_limit_dataset(splits["train"], MAX_TRAIN_SAMPLES, STREAMING)
    eval_source = maybe_limit_dataset(splits["eval"], MAX_EVAL_SAMPLES, STREAMING)

    train_dataset = DatasetClass(
        train_source,
        split_name=splits["train_split_name"],
        eval_case=splits["eval_case"],
        transform=clip_preprocess,
        task_type="classification",
    )
    eval_dataset = DatasetClass(
        eval_source,
        split_name=splits["eval_split_name"],
        eval_case=splits["eval_case"],
        transform=clip_preprocess,
        task_type="classification",
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False if STREAMING else True,
        num_workers=NUM_WORKERS,
        collate_fn=collate_unified_batch,
    )
    eval_loader = DataLoader(
        eval_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_unified_batch,
    )
    return train_loader, eval_loader

## 6. Helper trích CLIP embedding

In [ ]:
def extract_clip_features(loader, split_label):
    features = []
    labels = []
    rows = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Extract CLIP features: {split_label}"):
            images = batch["image"].to(DEVICE)
            image_features = clip_model.encode_image(images)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            image_features = image_features.float().cpu().numpy()

            features.append(image_features)
            labels.append(batch["label"].numpy())
            for meta in batch["metadata"]:
                rows.append(meta)

    x = np.concatenate(features, axis=0)
    y = np.concatenate(labels, axis=0).astype(int)
    meta_df = pd.DataFrame(rows)
    return x, y, meta_df


## 7. Helper train Logistic Regression tầng cuối

In [ ]:
def train_linear_probe(x_train, y_train):
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            C=1.0,
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_SEED,
        )),
    ])
    clf.fit(x_train, y_train)
    return clf

## 8. Helper evaluate và lưu kết quả

In [ ]:
def compute_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_prob))
        metrics["average_precision"] = float(average_precision_score(y_true, y_prob))
    else:
        metrics["roc_auc"] = None
        metrics["average_precision"] = None
    return metrics


def evaluate_by_generator(pred_df):
    rows = []
    for generator, part in pred_df.groupby("generator"):
        metrics = compute_metrics(part["label"].to_numpy(), part["fake_probability"].to_numpy())
        metrics["generator"] = generator
        metrics["num_samples"] = int(len(part))
        rows.append(metrics)
    return pd.DataFrame(rows).sort_values("generator")


def evaluate_and_save(exp, splits, run_dir, clf, x_train, y_train, x_eval, y_eval, eval_meta_df):
    y_prob = clf.predict_proba(x_eval)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    pred_df = eval_meta_df.copy()
    pred_df["label"] = y_eval
    pred_df["predicted_label"] = y_pred
    pred_df["fake_probability"] = y_prob
    pred_df["model_name"] = "clip_linear_probe"
    pred_df["experiment_name"] = exp["name"]
    pred_df["clip_model"] = CLIP_MODEL_NAME
    pred_df["clip_pretrained"] = CLIP_PRETRAINED

    overall_metrics = compute_metrics(y_eval, y_prob)
    overall_metrics.update({
        "experiment_name": exp["name"],
        "eval_case": exp["eval_case"],
        "generator": exp.get("generator"),
        "heldout_generator": exp.get("heldout_generator"),
        "base_generator": exp.get("base_generator"),
        "streaming": STREAMING,
        "balance_real": BALANCE_REAL,
        "max_train_samples": MAX_TRAIN_SAMPLES,
        "max_eval_samples": MAX_EVAL_SAMPLES,
        "clip_model_name": CLIP_MODEL_NAME,
        "clip_pretrained": CLIP_PRETRAINED,
        "train_rows_used": int(len(y_train)),
        "eval_rows_used": int(len(y_eval)),
        "split_info": {key: value for key, value in splits.items() if key not in {"train", "eval"}},
    })

    generator_metrics_df = evaluate_by_generator(pred_df)
    pred_df.to_csv(run_dir / "predictions" / "predictions.csv", index=False)
    generator_metrics_df.to_csv(run_dir / "metrics" / "generator_metrics.csv", index=False)
    with open(run_dir / "metrics" / "overall_metrics.json", "w", encoding="utf-8") as f:
        json.dump(overall_metrics, f, ensure_ascii=False, indent=2)
    return overall_metrics, generator_metrics_df, pred_df, y_prob

## 9. Helper plot ROC/confusion matrix

In [ ]:
def save_plots(run_dir, experiment_name, y_eval, y_prob, overall_metrics):
    if len(np.unique(y_eval)) == 2:
        fpr, tpr, _ = roc_curve(y_eval, y_prob)
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.plot(fpr, tpr, label=f'AUROC={overall_metrics["roc_auc"]:.3f}')
        ax.plot([0, 1], [0, 1], linestyle="--")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"CLIP Linear Probe ROC - {experiment_name}")
        ax.legend()
        fig.tight_layout()
        fig.savefig(run_dir / "plots" / "roc_curve.png", dpi=150)
        plt.close(fig)

    cm = np.array(overall_metrics["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["real", "fake"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["real", "fake"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(experiment_name)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(run_dir / "plots" / "confusion_matrix.png", dpi=150)
    plt.close(fig)

## 10. Chạy cả 4 trường hợp

In [ ]:
def make_run_dir(exp):
    run_dir = OUTPUT_ROOT / exp["name"] / RUN_ID
    for subdir in ["checkpoints", "features", "predictions", "metrics", "plots"]:
        (run_dir / subdir).mkdir(parents=True, exist_ok=True)
    return run_dir


def run_experiment(exp):
    run_dir = make_run_dir(exp)
    splits = build_splits_for_experiment(exp)
    train_loader, eval_loader = build_loaders(splits)

    x_train, y_train, train_meta_df = extract_clip_features(train_loader, f'{exp["name"]}/train')
    x_eval, y_eval, eval_meta_df = extract_clip_features(eval_loader, f'{exp["name"]}/eval')

    np.savez_compressed(run_dir / "features" / "train_clip_features.npz", features=x_train, labels=y_train)
    np.savez_compressed(run_dir / "features" / "eval_clip_features.npz", features=x_eval, labels=y_eval)
    train_meta_df.to_csv(run_dir / "features" / "train_metadata.csv", index=False)
    eval_meta_df.to_csv(run_dir / "features" / "eval_metadata.csv", index=False)

    clf = train_linear_probe(x_train, y_train)
    joblib.dump(clf, run_dir / "checkpoints" / "clip_logistic_regression.joblib")

    overall_metrics, generator_metrics_df, pred_df, y_prob = evaluate_and_save(
        exp=exp,
        splits=splits,
        run_dir=run_dir,
        clf=clf,
        x_train=x_train,
        y_train=y_train,
        x_eval=x_eval,
        y_eval=y_eval,
        eval_meta_df=eval_meta_df,
    )
    save_plots(run_dir, exp["name"], y_eval, y_prob, overall_metrics)

    print("\nDone:", exp["name"])
    print("RUN_DIR:", run_dir)
    print(json.dumps(overall_metrics, ensure_ascii=False, indent=2))
    display(generator_metrics_df)
    return overall_metrics


all_metrics = []
for exp in EXPERIMENT_CONFIGS:
    all_metrics.append(run_experiment(exp))

summary_df = pd.DataFrame(all_metrics)
summary_path = OUTPUT_ROOT / f"summary_metrics_{RUN_ID}.csv"
summary_df.to_csv(summary_path, index=False)
print("Summary saved:", summary_path)
display(summary_df[["experiment_name", "eval_case", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "average_precision", "train_rows_used", "eval_rows_used"]])

## 11. Ghi chú

Notebook này chạy cả 4 case trong `EXPERIMENT_CONFIGS`. Nếu chỉ muốn kiểm tra pipeline, giữ `MAX_TRAIN_SAMPLES` và `MAX_EVAL_SAMPLES` nhỏ. Khi chạy baseline thật, đặt cả hai về `None`.